In [46]:
import swisseph as swe
from datetime import datetime, timedelta

class DailyPanchangaEvents:
    """
    Calculates all Tithi and Nakshatra events (start/end times) occurring 
    within a single 24-hour period for specified Indian locations.
    """

    # --- CONSTANTS ---
    LOCATION_DATA = {
        "DELHI":  {"lat": 28.6139, "lon": 77.2090, "tz_offset": 5.5}, 
        "MUMBAI": {"lat": 19.0760, "lon": 72.8777, "tz_offset": 5.5},
        "UJJAIN": {"lat": 23.1765, "lon": 75.7885, "tz_offset": 5.5},
    }

    # 30 tithis: 0..14 = Shukla (15th = Purnima), 15..29 = Krishna (30th = Amavasya)
    TITHI_NAMES = [
        "Pratipada", "Dvitiya", "Tritiya", "Chaturthi", "Panchami", 
        "Shashthi", "Saptami", "Ashtami", "Navami", "Dashami", 
        "Ekadashi", "Dwadashi", "Trayodashi", "Chaturdashi", 
        "Purnima",
        "Pratipada", "Dvitiya", "Tritiya", "Chaturthi", "Panchami", 
        "Shashthi", "Saptami", "Ashtami", "Navami", "Dashami", 
        "Ekadashi", "Dwadashi", "Trayodashi", "Chaturdashi", "Amavasya"
    ]

    # 27 nakshatras: indices 0..26
    NAKSHATRA_NAMES = [
        "Ashwini", "Bharani", "Krittika", "Rohini", "Mrigashira", "Ardra", 
        "Punarvasu", "Pushya", "Ashlesha", "Magha", "Purva Phalguni", 
        "Uttara Phalguni", "Hasta", "Chitra", "Swati", "Vishakha", "Anuradha", 
        "Jyeshtha", "Mula", "Purva Ashadha", "Uttara Ashadha", "Shravana", 
        "Dhanishtha", "Shatabhisha", "Purva Bhadrapada", "Uttara Bhadrapada", "Revati"
    ]

    TITHI_SPAN = 12.0              # degrees of elongation per tithi
    NAKSHATRA_SPAN = 360.0 / 27.0  # degrees per nakshatra
    SECONDS_IN_JD = 86400.0        # seconds in one Julian day

    def __init__(self, ayanamsa=swe.SIDM_LAHIRI):
        # Use Lahiri ayanamsa by default (common in Indian panchang)
        swe.set_sid_mode(ayanamsa)

    # ----------------- LOW-LEVEL ASTRONOMY -----------------

    def _get_planet_lon(self, jd_ut: float, planet_id: int) -> float:
        """
        Returns sidereal ecliptic longitude of the given planet (degrees).
        """
        result, _ = swe.calc_ut(jd_ut, planet_id, swe.FLG_SIDEREAL)
        return result[0]  # longitude in degrees

    def _get_tithi_nakshatra_indices(self, jd_ut: float) -> tuple[int, int]:
        """
        Computes:
          - tithi index (0..29)
          - nakshatra index (0..26)
        at the given Julian Day (UT).
        """
        sun_lon = self._get_planet_lon(jd_ut, swe.SUN)
        moon_lon = self._get_planet_lon(jd_ut, swe.MOON)

        # Tithi by elongation (Moon - Sun), 12° per tithi
        elongation = (moon_lon - sun_lon) % 360.0
        tithi_index = int(elongation // self.TITHI_SPAN)  # 0..29

        # Nakshatra by Moon's longitude, 13°20' per segment
        nakshatra_index = int(moon_lon // self.NAKSHATRA_SPAN)  # 0..26

        return tithi_index, nakshatra_index

    # ----------------- TIME CONVERSIONS -----------------

    def _jd_to_local(self, jd_ut: float, tz_offset: float) -> datetime:
        """
        Converts Julian Day UT to local datetime using a fixed timezone offset.
        """
        # swe.revjul gives (year, month, day, hour_float_UT)
        year, month, day, hour_ut_float = swe.revjul(jd_ut)

        h = int(hour_ut_float)
        m = int((hour_ut_float - h) * 60)
        s = int(((hour_ut_float - h) * 60 - m) * 60)

        dt_utc = datetime(year, month, day, h, m, s)
        dt_local = dt_utc + timedelta(hours=tz_offset)
        return dt_local

    def _get_jd_bounds(self, date_str: str, tz_offset: float) -> tuple[float, float]:
        """
        Returns (jd_start, jd_end) for the *local* civil day date_str 
        (00:00 to 24:00 local time) converted to UT Julian Days.
        """
        # Local midnight start
        dt_start_local = datetime.strptime(date_str, "%Y-%m-%d")
        dt_end_local = dt_start_local + timedelta(days=1)

        # Convert local -> UTC
        dt_start_utc = dt_start_local - timedelta(hours=tz_offset)
        dt_end_utc = dt_end_local - timedelta(hours=tz_offset)

        jd_start = swe.julday(
            dt_start_utc.year,
            dt_start_utc.month,
            dt_start_utc.day,
            dt_start_utc.hour + dt_start_utc.minute / 60.0 + dt_start_utc.second / 3600.0
        )
        jd_end = swe.julday(
            dt_end_utc.year,
            dt_end_utc.month,
            dt_end_utc.day,
            dt_end_utc.hour + dt_end_utc.minute / 60.0 + dt_end_utc.second / 3600.0
        )

        return jd_start, jd_end

    # ----------------- ROOT FINDING FOR EVENT BOUNDARIES -----------------

    def _find_next_event_time(self, jd_start: float, event_type: str, current_index: int) -> float:
        """
        Finds the Julian Day when the next tithi or nakshatra starts 
        (i.e., index increments), using:
          - coarse search: 10 minute step
          - refine search: 1 second step
        """
        if event_type == 'TITHI':
            next_index = (current_index + 1) % 30
            checker = lambda jd: self._get_tithi_nakshatra_indices(jd)[0]
        elif event_type == 'NAKSHATRA':
            next_index = (current_index + 1) % 27
            checker = lambda jd: self._get_tithi_nakshatra_indices(jd)[1]
        else:
            return float('inf')

        coarse_step_jd = 10.0 / 1440.0  # 10 minutes in days
        jd_current = jd_start
        jd_limit = jd_start + 1.5       # don't search forever

        # Coarse step search
        while jd_current < jd_limit:
            jd_next_coarse = jd_current + coarse_step_jd
            if checker(jd_next_coarse) == next_index:
                break
            jd_current = jd_next_coarse

        if jd_current >= jd_limit:
            return float('inf')

        # Refine by seconds around the crossing
        jd_refine_start = jd_current - coarse_step_jd
        second_step_jd = 1.0 / self.SECONDS_IN_JD
        jd_refine = jd_refine_start

        while jd_refine < jd_limit:
            jd_refine_next = jd_refine + second_step_jd
            if checker(jd_refine_next) == next_index:
                return jd_refine_next
            jd_refine = jd_refine_next

        return float('inf')

    # ----------------- MAIN DAILY EVENT SCAN -----------------

    def _get_daily_events(self, jd_start: float, jd_end: float, event_type: str) -> list[dict]:
        events = []
        jd_current_start = jd_start
        tz_offset = self.LOCATION_DATA[self.location_name]["tz_offset"]

        while jd_current_start < jd_end:
            current_tithi_idx, current_nak_idx = self._get_tithi_nakshatra_indices(jd_current_start)

            if event_type == 'TITHI':
                current_index = current_tithi_idx
                name_list = self.TITHI_NAMES
            else:
                current_index = current_nak_idx
                name_list = self.NAKSHATRA_NAMES

            jd_next_event = self._find_next_event_time(jd_current_start, event_type, current_index)
            jd_event_end = min(jd_next_event, jd_end)

            start_dt = self._jd_to_local(jd_current_start, tz_offset)
            end_dt = self._jd_to_local(jd_event_end, tz_offset)

            if event_type == 'TITHI':
                # Paksha label (still useful even for Purnima / Amavasya)
                paksha = "Shukla" if current_tithi_idx < 15 else "Krishna"

                base_name = name_list[current_index]
                # Avoid duplicated strings like "Purnima (Purnima (Shukla))"
                if current_tithi_idx == 14:
                    display_name = "Purnima"
                elif current_tithi_idx == 29:
                    display_name = "Amavasya"
                else:
                    display_name = f"{base_name} ({paksha})"
            else:
                paksha = None
                display_name = name_list[current_index]

            events.append({
                "Element": event_type.capitalize(),
                "Name": display_name,
                "Paksha": paksha,
                "Start_Time": start_dt.strftime("%H:%M:%S"),
                "End_Time":   end_dt.strftime("%H:%M:%S"),
            })

            if jd_next_event >= jd_end:
                break

            jd_current_start = jd_next_event

        return events

    # ----------------- PUBLIC API -----------------

    def get_daily_panchanga2(self, date_str: str, location_name: str) -> dict:
        """
        Returns all Tithi and Nakshatra events for the given local civil date
        and supported location, along with start/end times in local time.
        """
        self.location_name = location_name.upper()
        if self.location_name not in self.LOCATION_DATA:
            return {"Error": f"Location must be one of: {list(self.LOCATION_DATA.keys())}"}

        try:
            # Validate date format
            datetime.strptime(date_str, "%Y-%m-%d")

            loc_data = self.LOCATION_DATA[self.location_name]
            tz_offset = loc_data["tz_offset"]

            jd_start, jd_end = self._get_jd_bounds(date_str, tz_offset)

            tithi_events = self._get_daily_events(jd_start, jd_end, 'TITHI')
            nakshatra_events = self._get_daily_events(jd_start, jd_end, 'NAKSHATRA')

            return {
                "Date": date_str,
                "Location": self.location_name.capitalize(),
                "Timezone": f"IST (UTC +{tz_offset})",
                "Tithi_Events": tithi_events,
                "Nakshatra_Events": nakshatra_events,
            }

        except ValueError as e:
            return {"Error": f"Input validation error: {e}. Ensure date is 'YYYY-MM-DD'."}
        except Exception as e:
            return {"Error": f"An unexpected error occurred: {type(e).__name__} - {e}"}

In [49]:
calc = DailyPanchangaEvents()
res = calc.get_daily_panchanga2("2025-12-09", "DELHI")
res

{'Date': '2025-12-09',
 'Location': 'Delhi',
 'Timezone': 'IST (UTC +5.5)',
 'Tithi_Events': [{'Element': 'Tithi',
   'Name': 'Panchami (Krishna)',
   'Paksha': 'Krishna',
   'Start_Time': '00:00:00',
   'End_Time': '14:29:26'},
  {'Element': 'Tithi',
   'Name': 'Shashthi (Krishna)',
   'Paksha': 'Krishna',
   'Start_Time': '14:29:26',
   'End_Time': '00:00:00'}],
 'Nakshatra_Events': [{'Element': 'Nakshatra',
   'Name': 'Pushya',
   'Paksha': None,
   'Start_Time': '00:00:00',
   'End_Time': '02:52:51'},
  {'Element': 'Nakshatra',
   'Name': 'Ashlesha',
   'Paksha': None,
   'Start_Time': '02:52:51',
   'End_Time': '00:00:00'}]}